# Agent高级用法

## 1.设置Agent名称

使用字段name来进行设置

举例：

In [ ]:
import os

from accelerate.commands.config.update import description
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent
from rich import print as rprint

# 1、读取.env配置文件中的信息；相关的环境变量要以.env中环境变量优先
load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
model = ChatDeepSeek(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
)

agent = create_agent(
    model = model,
    name = "agent111"
)


res = agent.invoke({
    "messages":[
        {"role":"system", "content":"你是一个数学老师，擅长以通俗易懂的方式讲解数学问题！"},
        {"role":"user", "content":""}
    ]
})

rprint(res)

for msg in res["messages"]:
    msg.pretty_print()   # 展示名称

## 2.系统提示词的设置

使用system_prompt参数设置，可以是str，也可以是SystemMessage

举例：

In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent
from langchain_tavily import TavilySearch
from rich import print as rprint

# 读取.env配置文件中的信息；相关的环境变量要以.env中环境变量优先
load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 1、模型的初始化
model = ChatDeepSeek(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
)


# 2.导入工具
web_search = TavilySearch(max_results=2)

# 3.创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"    # 不会在result结果中显示
)

# 4.运行Agent获得结果
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "请帮我查询2026年足球世界杯是哪个国家举办的？"}
    ]}
)

rprint(result)

举例2：

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage

flag = 0

@tool
def get_weather(city: str):
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    global flag
    flag += 1

    if flag < 3:
        # raise Exception("暂时无法访问")
        return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试"

    return f"{city}今天天气挺好"


messages = [
    HumanMessage("你好，杭州今天的天气如何？")
]

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=SystemMessage(
        "你是一个天气助手。"
        "当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，"
        "说明是临时故障，不要立即放弃；"
        "你应再次调用同一个工具，最多重试 3 次。"
        "如果 3 次后仍失败，再向用户说明服务暂时不可用。"
    )
)
response = agent.invoke({"messages": messages})
# print(response)
for msg in response["messages"]:
    msg.pretty_print()

## 3.结构化输出的四种策略

### 3.1ProviderStrategy策略：



In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent
from rich import print as rprint
from langchain.agents.structured_output import ProviderStrategy
from pydantic import BaseModel, Field

# 读取.env配置文件中的信息；相关的环境变量要以.env中环境变量优先
load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 1、模型的初始化
model = ChatDeepSeek(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
    extra_body={"thinking":{"type":"disabled"}}
)

# 使用Pydantic结构化的方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户的姓名")
    email: str = Field(description="用户的邮箱")
    phone: str = Field(description="用户的电话")


agent = create_agent(
    model = model,
    response_format=ProviderStrategy(schema=ContractInfo)
)

res = agent.invoke({
    "messages":[
        {"role":"user", "content":"从以下信息中，提取用户信息：小明的邮箱是111@136.com,电话是124588"}
    ]
})

rprint(res)

# ProviderStrategy 依赖模型提供方原生支持 JSON Schema 结构化输出（OpenAI 的 response_format={"type": "json_schema", ...} 那套）。

# DeepSeek 官方 API 目前对 json_schema 的支持是有限的（支持 json_object，但对严格的 schema 约束不如 OpenAI 完整）。如果 DeepSeek 这边不认，ProviderStrategy 就可能失败。

### 3.2 ToolStrategy策略

In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent
from rich import print as rprint
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field

# 读取.env配置文件中的信息；相关的环境变量要以.env中环境变量优先
load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 1、模型的初始化
model = ChatDeepSeek(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
    extra_body = {"thinking":{"type":"disabled"}}
)

# 使用Pydantic结构化的方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户的姓名")
    email: str = Field(description="用户的邮箱")
    phone: str = Field(description="用户的电话")


agent = create_agent(
    model = model,
    response_format=ToolStrategy(schema=ContractInfo)
)

res = agent.invoke({
    "messages":[
        {"role":"user", "content":"从以下信息中，提取用户信息：小明的邮箱是111@136.com,电话是124588"}
    ]
})

rprint(res)

### 3.2 AutoStrategy / Type策略：

举例：

In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent
from rich import print as rprint
from langchain.agents.structured_output import AutoStrategy
from pydantic import BaseModel, Field

# 读取.env配置文件中的信息；相关的环境变量要以.env中环境变量优先
load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 1、模型的初始化
model = ChatDeepSeek(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
    extra_body = {"thinking":{"type":"disabled"}}
)

# 使用Pydantic结构化的方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户的姓名")
    email: str = Field(description="用户的邮箱")
    phone: str = Field(description="用户的电话")


agent = create_agent(
    model = model,
    response_format=AutoStrategy(schema=ContractInfo)
    # response_format=ContractInfo  -- 不推荐这样写
)

res = agent.invoke({
    "messages":[
        {"role":"user", "content":"从以下信息中，提取用户信息：小明的邮箱是111@136.com,电话是124588"}
    ]
})

rprint(res)